# Zazo — تحويل أوزان LoRA إلى Android

هذا الدفتر لا يدرب النموذج. يأخذ `zazo_lora_weights.zip` الناتج من دفتر التدريب، ثم يحول أوزان Hugging Face إلى حزمة Android يقبلها التطبيق.

الناتج النهائي يجب أن يحتوي بالضبط على:
```text
base.bin
lora.bin
```

يلزم GPU و`HF_TOKEN` للجلسة الحالية، ومساحة تخزين كبيرة لأن نموذج Gemma الأساسي يُنزّل مؤقتًا.

In [ ]:
%pip install -q -U mediapipe huggingface_hub safetensors
import os, json, shutil, zipfile, getpass, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('اختر GPU قبل تشغيل التحويل.')
print('GPU:', torch.cuda.get_device_name(0))
BASE_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
WORK_DIR = BASE_DIR / 'zazo_android_export'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('مجلد العمل:', WORK_DIR)

In [ ]:
# ضع التوكن للجلسة الحالية فقط؛ لن يتم حفظه في Drive أو داخل الدفتر.
HF_TOKEN = ""  # مثال: hf_xxxxxxxxxxxxxxxxx
if not HF_TOKEN.strip():
    HF_TOKEN = getpass.getpass('HF_TOKEN من Hugging Face (يبدأ بـ hf_): ').strip()
if not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('ضع توكن Hugging Face صحيحًا يبدأ بـ hf_.')
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
MODEL_NAME = 'google/gemma-2b-it'
print('تم تسجيل الدخول للجلسة الحالية فقط')

In [ ]:
SUPPORTED = {'.zip', '.safetensors', '.json'}
def find_adapter_input():
    roots = [Path('/kaggle/input'), Path('/content')]
    candidates = [p for root in roots if root.exists() for p in root.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED and ('zazo' in p.name.lower() or 'adapter' in p.name.lower())]
    if candidates:
        return max(candidates, key=lambda p: p.stat().st_mtime)
    try:
        from google.colab import files
        print('اختر zazo_lora_weights.zip')
        uploaded = files.upload()
        for name, content in uploaded.items():
            path = Path('/content') / Path(name).name
            path.write_bytes(content)
            return path
    except ImportError:
        pass
    raise RuntimeError('ارفع zazo_lora_weights.zip إلى Kaggle كـDataset أو اختره من Colab.')

ADAPTER_INPUT = find_adapter_input()
ADAPTER_DIR = WORK_DIR / 'adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
if ADAPTER_INPUT.suffix.lower() == '.zip':
    with zipfile.ZipFile(ADAPTER_INPUT) as archive:
        archive.extractall(ADAPTER_DIR)
else:
    shutil.copy2(ADAPTER_INPUT, ADAPTER_DIR / ADAPTER_INPUT.name)

adapter_weights = next(ADAPTER_DIR.rglob('adapter_model.safetensors'), None)
adapter_config = next(ADAPTER_DIR.rglob('adapter_config.json'), None)
if adapter_weights is None or adapter_config is None:
    raise RuntimeError('الملف يجب أن يحتوي adapter_model.safetensors وadapter_config.json من دفتر التدريب.')
config = json.loads(adapter_config.read_text(encoding='utf-8'))
LORA_R = int(config.get('r', config.get('lora_r', 8)))
print('ملف LoRA:', adapter_weights)
print('رتبة LoRA:', LORA_R)

In [ ]:
from huggingface_hub import snapshot_download
from mediapipe.tasks.python.genai import converter

BASE_MODEL_DIR = WORK_DIR / 'gemma_base'
if not (BASE_MODEL_DIR / 'config.json').exists():
    print('جاري تنزيل نموذج Gemma الأساسي...')
    snapshot_download(
        repo_id=MODEL_NAME, local_dir=str(BASE_MODEL_DIR), token=HF_TOKEN,
        allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt'],
    )

ANDROID_DIR = WORK_DIR / 'android'
ANDROID_DIR.mkdir(parents=True, exist_ok=True)
base_output = ANDROID_DIR / 'base.bin'
lora_output = ANDROID_DIR / 'lora.bin'
intermediate = ANDROID_DIR / 'intermediate'
conversion_config = converter.ConversionConfig(
    input_ckpt=str(BASE_MODEL_DIR), ckpt_format='safetensors',
    model_type='GEMMA_2B', backend='gpu', output_dir=str(intermediate),
    combine_file_only=False, vocab_model_file=str(BASE_MODEL_DIR),
    output_tflite_file=str(base_output), lora_ckpt=str(adapter_weights),
    lora_rank=LORA_R, lora_output_tflite_file=str(lora_output),
)
converter.convert_checkpoint(conversion_config)
if not base_output.is_file() or not lora_output.is_file():
    raise RuntimeError('فشل التحويل ولم يتم إنتاج base.bin وlora.bin.')
print('تم إنتاج base.bin وlora.bin')

In [ ]:
ZIP_PATH = BASE_DIR / 'zazo_android_model.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    archive.write(ANDROID_DIR / 'base.bin', 'base.bin')
    archive.write(ANDROID_DIR / 'lora.bin', 'lora.bin')
print('الحزمة الجاهزة للتطبيق:', ZIP_PATH)

try:
    from google.colab import files, drive
    files.download(str(ZIP_PATH))
    drive.mount('/content/drive', force_remount=False)
    drive_path = Path('/content/drive/MyDrive/zazo_android_model.zip')
    shutil.copy2(ZIP_PATH, drive_path)
    print('تم نسخها إلى Google Drive:', drive_path)
except ImportError:
    print('في Kaggle نزّل الملف من تبويب Output:', ZIP_PATH)
except Exception as error:
    print('تم إنتاج الملف محليًا، وتعذر نسخ Drive:', error)